# Relationship Score Analysis - CORRECTED

## Purpose
This notebook trains a Relationship model using the features specified in the flowchart:
1. `days_since_last_contact` (calculated from `last_contact_date`)
2. `sentiment_score`
3. `sentiment_category`
4. `licenses_total`
5. `licenses_used`
6. `utilization_percentage`

**IMPORTANT**: Following the flowchart, the Relationship model uses these 6 features only!

---

## Input features (what the model takes)

| # | **Raw feature** | **Description** | **After encoding** |
|---|-----------------|-----------------|--------------------|
| 1 | `days_since_last_contact` | Days since last contact (from `last_contact_date`) | Same (numeric) |
| 2 | `sentiment_score` | Numeric sentiment in [-1, 1] | Same (numeric) |
| 3 | `sentiment_category` | Category: neutral, positive, very_positive, negative, very_negative | One-hot: `sentiment_neutral`, `sentiment_positive`, `sentiment_very_negative`, `sentiment_very_positive` (4 dummies) |
| 4 | `licenses_total` | Total number of licenses | Same (numeric) |
| 5 | `licenses_used` | Number of licenses in use | Same (numeric) |
| 6 | `utilization_percentage` | Utilization % (e.g. 25–95) | Same (numeric) |

**Target:** `relationship_score` (continuous; the model is a **regressor**, not a classifier).

**Total model inputs:** 9 columns after one-hot encoding (5 numeric + 4 sentiment dummies).

In [66]:
# Install required packages
!uv add pandas numpy matplotlib seaborn openpyxl scikit-learn xgboost lightgbm catboost joblib

Resolved 109 packages in 2ms
Audited 85 packages in 21ms


In [67]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime, timedelta
import warnings
import joblib
import json

# Machine Learning Libraries
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.feature_selection import SelectKBest, f_regression, mutual_info_regression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Advanced Models
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from sklearn.ensemble import (
    RandomForestRegressor, 
    GradientBoostingRegressor,
    StackingRegressor,
    VotingRegressor
)
from sklearn.linear_model import RidgeCV, ElasticNetCV
from sklearn.svm import SVR
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV

warnings.filterwarnings('ignore')

# Set style for better visualizations
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 7)
plt.rcParams['font.size'] = 10

print("Libraries imported successfully!")

Libraries imported successfully!


## 1. Load and Explore Data

In [68]:
# Resolve path to Research/customer_data_25000.xlsx (same as other EDA notebooks)
cwd = Path.cwd()
research = cwd if (cwd / 'customer_data_25000.xlsx').exists() else (cwd.parent if (cwd.parent / 'customer_data_25000.xlsx').exists() else cwd / 'Research')
if not (research / 'customer_data_25000.xlsx').exists():
    research = Path(r"D:\Projects_Main\Renewal-Upsell-Advisor\Research")
data_path = research / 'customer_data_25000.xlsx'
if not data_path.exists():
    raise FileNotFoundError("Could not find customer_data_25000.xlsx. Please check the file path.")

df = pd.read_excel(data_path, sheet_name="Accounts")

print(f"Dataset loaded successfully!")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"\nColumns in dataset:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2d}. {col}")

df.head()

Dataset loaded successfully!
Shape: 25,000 rows × 27 columns

Columns in dataset:
   1. name
   2. domain
   3. industry
   4. company_size
   5. arr
   6. mrr
   7. contract_start_date
   8. contract_end_date
   9. renewal_date
  10. last_contact_date
  11. status
  12. renewal_stage
  13. health_score
  14. risk_score
  15. relationship_score
  16. churn_probability
  17. sentiment_score
  18. sentiment_category
  19. licenses_total
  20. licenses_used
  21. utilization_percentage
  22. csm_name
  23. csm_email
  24. primary_contact_name
  25. primary_contact_email
  26. primary_contact_phone
  27. salesforce_id


,name,domain,industry,company_size,arr,mrr,contract_start_date,contract_end_date,renewal_date,last_contact_date,...,sentiment_category,licenses_total,licenses_used,utilization_percentage,csm_name,csm_email,primary_contact_name,primary_contact_email,primary_contact_phone,salesforce_id
0,Cole LLC,colellc.com,Technology,Medium,156049,13004.08,2025-06-12,2026-06-12,2026-06-12,2026-02-14 10:30:11,...,negative,20,15,79,Emily Rodriguez,emily.rodriguez@company.com,Danielle Johnson,john21@example.net,001-581-896-0013x3890,SF-835392
1,"Stevens, Martinez and Nielsen",stevensmartinezandnielsen.com,Media,Enterprise,378618,31551.50,2025-05-31,2026-05-31,2026-05-31,2026-02-13 10:30:11,...,very_negative,30,23,79,Sarah Chen,sarah.chen@company.com,Lisa Smith,helenpeterson@example.org,651.216.1559,NaN
2,Clark-Adams,clark-adams.com,Manufacturing,Medium,62823,5235.25,2025-05-06,2026-05-06,2026-05-06,2026-02-07 10:30:11,...,very_positive,150,52,35,Emily Rodriguez,emily.rodriguez@company.com,Christian Carter,barbara10@example.net,441.731.6475,SF-148050
3,"Porter, Wilkerson and Day",porterwilkersonandday.com,Research,Medium,96054,8004.50,2025-06-15,2026-06-15,2026-06-15,2026-02-14 10:30:11,...,very_negative,200,142,71,Sarah Chen,sarah.chen@company.com,Sharon Wong,amandasanchez@example.com,(748)535-0305x6413,SF-356702
4,Carlson-Mcdonald,carlson-mcdonald.com,Consulting,Large,257691,21474.25,2025-04-12,2026-04-12,2026-04-12,2026-01-23 10:30:11,...,very_positive,100,32,32,James Wilson,james.wilson@company.com,Douglas Taylor,julie69@example.com,(332)887-1012x269,NaN


## 2. Feature Engineering (Following Flowchart)

**CRITICAL**: According to the flowchart, Relationship model should use:
1. `days_since_last_contact` (from `last_contact_date`)
2. `sentiment_score`
3. `sentiment_category`
4. `licenses_total`
5. `licenses_used`
6. `utilization_percentage`

In [69]:
# Feature Engineering - ONLY flowchart features
print("=" * 60)
print("STEP 1: Feature Engineering (Following Flowchart)")
print("=" * 60)
print("⚠️  IMPORTANT: Relationship model should use:")
print("  1. days_since_last_contact (from last_contact_date)")
print("  2. sentiment_score")
print("  3. sentiment_category")
print("  4. licenses_total")
print("  5. licenses_used")
print("  6. utilization_percentage")
print("=" * 60)

# Start with original dataframe
X_engineered = df.copy()

# Calculate days_since_last_contact from last_contact_date
print("\n📅 Calculating days_since_last_contact...")
if 'last_contact_date' in X_engineered.columns:
    # Convert to datetime if not already
    X_engineered['last_contact_date'] = pd.to_datetime(X_engineered['last_contact_date'], errors='coerce')
    
    # Calculate days since last contact (using max date as reference)
    reference_date = X_engineered['last_contact_date'].max()
    if pd.isna(reference_date):
        reference_date = pd.Timestamp.now()
    
    X_engineered['days_since_last_contact'] = (reference_date - X_engineered['last_contact_date']).dt.days
    X_engineered['days_since_last_contact'] = X_engineered['days_since_last_contact'].fillna(X_engineered['days_since_last_contact'].median())
    
    print(f"✓ Calculated days_since_last_contact")
    print(f"  Range: [{X_engineered['days_since_last_contact'].min():.0f}, {X_engineered['days_since_last_contact'].max():.0f}] days")
    print(f"  Mean: {X_engineered['days_since_last_contact'].mean():.2f} days")
    print(f"  Median: {X_engineered['days_since_last_contact'].median():.2f} days")
else:
    print("❌ ERROR: 'last_contact_date' column not found!")
    print(f"Available columns: {list(X_engineered.columns)}")
    raise ValueError("last_contact_date column is required for Relationship model")

# Verify all required features exist
required_cols = ['sentiment_score', 'sentiment_category', 'licenses_total', 'licenses_used', 'utilization_percentage']
missing_cols = [col for col in required_cols if col not in X_engineered.columns]

if missing_cols:
    print(f"❌ ERROR: Missing required columns: {missing_cols}")
    print(f"Available columns: {list(X_engineered.columns)}")
    raise ValueError(f"Missing required columns: {missing_cols}")

print(f"\n✓ All required features available:")
print(f"  sentiment_score: Range [{X_engineered['sentiment_score'].min():.2f}, {X_engineered['sentiment_score'].max():.2f}]")
print(f"  sentiment_category: {X_engineered['sentiment_category'].value_counts().to_dict()}")
print(f"  licenses_total: Range [{X_engineered['licenses_total'].min():.0f}, {X_engineered['licenses_total'].max():.0f}]")
print(f"  licenses_used: Range [{X_engineered['licenses_used'].min():.0f}, {X_engineered['licenses_used'].max():.0f}]")
print(f"  utilization_percentage: Range [{X_engineered['utilization_percentage'].min():.2f}, {X_engineered['utilization_percentage'].max():.2f}]")

# Select features according to flowchart
print("\n📊 Selecting features according to flowchart:")
print("  → days_since_last_contact (from last_contact_date)")
print("  → sentiment_score")
print("  → sentiment_category")
print("  → licenses_total")
print("  → licenses_used")
print("  → utilization_percentage")

# Verify all required features exist
required_features = ['days_since_last_contact', 'sentiment_score', 'sentiment_category', 'licenses_total', 'licenses_used', 'utilization_percentage']
missing_features = set(required_features) - set(X_engineered.columns)

if missing_features:
    print(f"\n❌ ERROR: Missing required features: {missing_features}")
    print(f"Available columns: {list(X_engineered.columns)}")
    raise ValueError(f"Missing required features: {missing_features}")

X_features = X_engineered[required_features].copy()

# Get target variable
if 'relationship_score' in df.columns:
    y = df['relationship_score'].copy()
    print("\n✓ Using original 'relationship_score' as target")
elif 'total_relationship_signal' in df.columns:
    y = df['total_relationship_signal'].copy()
    print("\n⚠️  Using 'total_relationship_signal' as target")
else:
    raise ValueError("Neither 'relationship_score' nor 'total_relationship_signal' found in data")

# Handle missing values
# Handle numeric columns with median
numeric_cols = X_features.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    X_features[col] = X_features[col].fillna(X_features[col].median())

# Handle categorical columns (sentiment_category) with mode
categorical_cols = X_features.select_dtypes(include=['object']).columns
for col in categorical_cols:
    mode_val = X_features[col].mode()
    if len(mode_val) > 0:
        X_features[col] = X_features[col].fillna(mode_val[0])
    else:
        X_features[col] = X_features[col].fillna('neutral')  # Default value

# Handle target variable
y = y.fillna(y.median())

print(f"\n✓ Feature engineering complete!")
print(f"✓ Using {len(X_features.columns)} features: {list(X_features.columns)}")
print(f"✓ Target variable: {y.name}")
print(f"✓ Final shape: {X_features.shape[0]:,} rows × {X_features.shape[1]} features")

X_features.head()

STEP 1: Feature Engineering (Following Flowchart)
⚠️  IMPORTANT: Relationship model should use:
  1. days_since_last_contact (from last_contact_date)
  2. sentiment_score
  3. sentiment_category
  4. licenses_total
  5. licenses_used
  6. utilization_percentage

📅 Calculating days_since_last_contact...
✓ Calculated days_since_last_contact
  Range: [0, 30] days
  Mean: 15.00 days
  Median: 15.00 days

✓ All required features available:
  sentiment_score: Range [-1.00, 1.00]
  sentiment_category: {'negative': 5019, 'positive': 5014, 'very_positive': 5005, 'neutral': 4996, 'very_negative': 4966}
  licenses_total: Range [10, 200]
  licenses_used: Range [2, 190]
  utilization_percentage: Range [25.00, 95.00]

📊 Selecting features according to flowchart:
  → days_since_last_contact (from last_contact_date)
  → sentiment_score
  → sentiment_category
  → licenses_total
  → licenses_used
  → utilization_percentage

✓ Using original 'relationship_score' as target

✓ Feature engineering complete!

,days_since_last_contact,sentiment_score,sentiment_category,licenses_total,licenses_used,utilization_percentage
0,7,-0.3818,negative,20,15,79
1,8,-0.8103,very_negative,30,23,79
2,14,0.7812,very_positive,150,52,35
3,7,-0.9050,very_negative,200,142,71
4,29,0.8049,very_positive,100,32,32


**Input features are fixed:** the model uses only the 6 flowchart features above. No extra inputs are added.

In [70]:
# Model uses ONLY the 6 flowchart features as inputs (no extra features).
print("✓ Input features for prediction: only the 6 flowchart features (days_since_last_contact, sentiment_score, sentiment_category, licenses_total, licenses_used, utilization_percentage)")

✓ Input features for prediction: only the 6 flowchart features (days_since_last_contact, sentiment_score, sentiment_category, licenses_total, licenses_used, utilization_percentage)


### Feature engineering (from the 6 inputs only)

Derived features are built **only from the 6 flowchart inputs**. No extra columns from the dataset. At prediction time, compute these same derived features from your 6 raw inputs.

In [71]:
# Derived features from the 6 inputs only (can help R²)
# 1. Sentiment × utilization (interaction)
X_features['sentiment_x_utilization'] = X_features['sentiment_score'] * (X_features['utilization_percentage'] / 100.0)
# 2. Licenses ratio (used/total; avoid div by zero)
X_features['licenses_ratio'] = np.where(
    X_features['licenses_total'] > 0,
    X_features['licenses_used'] / X_features['licenses_total'],
    0.0
)
# 3. Days bins: recent (0–7), mid (8–14), older (15+)
X_features['days_recent'] = (X_features['days_since_last_contact'] <= 7).astype(int)
X_features['days_mid'] = ((X_features['days_since_last_contact'] >= 8) & (X_features['days_since_last_contact'] <= 14)).astype(int)
print("✓ Engineered features (from 6 inputs only): sentiment_x_utilization, licenses_ratio, days_recent, days_mid")
print(f"  Total features now: {list(X_features.columns)}")
X_features.head()

✓ Engineered features (from 6 inputs only): sentiment_x_utilization, licenses_ratio, days_recent, days_mid
  Total features now: ['days_since_last_contact', 'sentiment_score', 'sentiment_category', 'licenses_total', 'licenses_used', 'utilization_percentage', 'sentiment_x_utilization', 'licenses_ratio', 'days_recent', 'days_mid']


,days_since_last_contact,sentiment_score,sentiment_category,licenses_total,licenses_used,utilization_percentage,sentiment_x_utilization,licenses_ratio,days_recent,days_mid
0,7,-0.3818,negative,20,15,79,-0.301622,0.750000,1,0
1,8,-0.8103,very_negative,30,23,79,-0.640137,0.766667,0,1
2,14,0.7812,very_positive,150,52,35,0.273420,0.346667,0,1
3,7,-0.9050,very_negative,200,142,71,-0.642550,0.710000,1,0
4,29,0.8049,very_positive,100,32,32,0.257568,0.320000,0,0


## 3. Data Preparation

In [72]:
# Data Preparation
print("=" * 60)
print("STEP 2: Data Preparation")
print("=" * 60)

# One-hot encode all categorical columns (explicit list so we never pass strings to the scaler)
print("\n📊 Encoding categorical features...")
cat_cols = ['sentiment_category'] + [c for c in ['company_size', 'industry'] if c in X_features.columns]
X_features_encoded = pd.get_dummies(X_features, columns=cat_cols, drop_first=True)

print(f"✓ One-hot encoded: {cat_cols}")
print(f"  Original features: {list(X_features.columns)}")
print(f"  Encoded features: {list(X_features_encoded.columns)}")

# Outlier removal: None = no removal (keeps all rows, best for R²). Use 2.0 or 1.5 only if you want to drop extremes.
IQR_MULTIPLIER = None
print("\n🔍 Removing outliers...")
original_shape = X_features_encoded.shape[0]

df_combined = X_features_encoded.copy()
df_combined['target'] = y.values

if IQR_MULTIPLIER is None:
    df_cleaned = df_combined.copy()
    print("✓ No outlier removal (all rows kept — best for R² with limited features)")
else:
    # IQR only on continuous columns (not 0/1 dummies or engineered) so we don't remove too many rows
    iqr_cols = [c for c in ['days_since_last_contact', 'sentiment_score', 'licenses_total', 'licenses_used', 'utilization_percentage', 'target'] if c in df_combined.columns]
    Q1 = df_combined[iqr_cols].quantile(0.25)
    Q3 = df_combined[iqr_cols].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - IQR_MULTIPLIER * IQR
    upper_bound = Q3 + IQR_MULTIPLIER * IQR
    outlier_mask = pd.Series([True] * len(df_combined))
    for col in iqr_cols:
        outlier_mask = outlier_mask & ((df_combined[col] >= lower_bound[col]) & (df_combined[col] <= upper_bound[col]))
    df_cleaned = df_combined[outlier_mask].copy()
    outliers_removed = original_shape - len(df_cleaned)
    print(f"✓ Removed {outliers_removed:,} outlier rows (IQR × {IQR_MULTIPLIER}, {outliers_removed/original_shape*100:.2f}%)")

X_features_cleaned = df_cleaned.drop(columns=['target'])
y_cleaned = df_cleaned['target'].copy()
print(f"✓ Remaining: {len(df_cleaned):,} rows (from {original_shape:,})")


# Train-test split
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_features_cleaned, y_cleaned, test_size=0.2, random_state=42, shuffle=True
)

print(f"✓ Train set: {len(X_train):,} samples")
print(f"✓ Test set: {len(X_test):,} samples")

# Scaling (RobustScaler for outlier resistance)
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrame
X_train_df = pd.DataFrame(X_train_scaled, columns=X_features_encoded.columns, index=X_train.index)
X_test_df = pd.DataFrame(X_test_scaled, columns=X_features_encoded.columns, index=X_test.index)

print(f"\n✓ Scaling completed using RobustScaler")
print(f"✓ Train shape: {X_train_df.shape}")
print(f"✓ Test shape: {X_test_df.shape}")
print(f"✓ Total features after encoding: {X_train_df.shape[1]}")

STEP 2: Data Preparation

📊 Encoding categorical features...
✓ One-hot encoded: ['sentiment_category']
  Original features: ['days_since_last_contact', 'sentiment_score', 'sentiment_category', 'licenses_total', 'licenses_used', 'utilization_percentage', 'sentiment_x_utilization', 'licenses_ratio', 'days_recent', 'days_mid']
  Encoded features: ['days_since_last_contact', 'sentiment_score', 'licenses_total', 'licenses_used', 'utilization_percentage', 'sentiment_x_utilization', 'licenses_ratio', 'days_recent', 'days_mid', 'sentiment_category_neutral', 'sentiment_category_positive', 'sentiment_category_very_negative', 'sentiment_category_very_positive']

🔍 Removing outliers...


✓ No outlier removal (all rows kept — best for R² with limited features)
✓ Remaining: 25,000 rows (from 25,000)
✓ Train set: 20,000 samples
✓ Test set: 5,000 samples

✓ Scaling completed using RobustScaler
✓ Train shape: (20000, 13)
✓ Test shape: (5000, 13)
✓ Total features after encoding: 13


## 4a. Train each model with hyperparameter tuning

Train all base models first with `RandomizedSearchCV`; then stack them in the next section.

In [ ]:
# Train Multiple Models (input features = flowchart only; no extra inputs)
print("=" * 60)
print("STEP 3: Model Training")
print("=" * 60)

# Use raw target (log-target was tried; with 6 inputs only, raw often matches or beats it)
USE_LOG_TARGET = False
y_train_fit = np.log1p(y_train) if USE_LOG_TARGET else y_train
def _inv_pred(p):
    return np.expm1(p) if USE_LOG_TARGET else p

models = {}
results = {}
best_params = {}
results = {}

# 1. XGBoost with Hyperparameter Tuning
print("\n1. Training XGBoost with Hyperparameter Tuning...")
xgb_base = XGBRegressor(random_state=42, n_jobs=-1)
xgb_params = {
    'n_estimators': [200, 300, 500],
    'learning_rate': [0.05, 0.1, 0.15],
    'max_depth': [5, 7, 9],
    'subsample': [0.8, 0.9],
    'colsample_bytree': [0.8, 0.9],
    'reg_alpha': [0.1, 0.5],
    'reg_lambda': [1.0, 2.0]
}
xgb_search = RandomizedSearchCV(
    xgb_base, xgb_params, n_iter=40, cv=5, 
    scoring='r2', n_jobs=-1, random_state=42, verbose=1
)
xgb_search.fit(X_train_df, y_train_fit)
xgb = xgb_search.best_estimator_
models['XGBoost'] = xgb
best_params['XGBoost'] = xgb_search.best_params_
y_pred_xgb = _inv_pred(xgb.predict(X_test_df))
results['XGBoost'] = {
    'r2': r2_score(y_test, y_pred_xgb),
    'mae': mean_absolute_error(y_test, y_pred_xgb),
    'rmse': np.sqrt(mean_squared_error(y_test, y_pred_xgb))
}
print(f"   Best params: {xgb_search.best_params_}")
print(f"   R²: {results['XGBoost']['r2']:.4f}, MAE: {results['XGBoost']['mae']:.4f}, RMSE: {results['XGBoost']['rmse']:.4f}")

# 2. LightGBM
print("\n2. Training LightGBM with Hyperparameter Tuning...")
print("\n2. Training LightGBM with Hyperparameter Tuning...")
lgb_base = LGBMRegressor(random_state=42, n_jobs=-1, verbose=-1)
lgb_params = {
    'n_estimators': [200, 300, 500],
    'learning_rate': [0.05, 0.1, 0.15],
    'max_depth': [5, 7, 9],
    'num_leaves': [31, 50, 100],
    'subsample': [0.8, 0.9],
    'colsample_bytree': [0.8, 0.9],
    'reg_alpha': [0.1, 0.5],
    'reg_lambda': [1.0, 2.0]
}
lgb_search = RandomizedSearchCV(
    lgb_base, lgb_params, n_iter=40, cv=5,
    scoring='r2', n_jobs=-1, random_state=42, verbose=1
)
lgb_search.fit(X_train_df, y_train_fit)
lgb = lgb_search.best_estimator_
models['LightGBM'] = lgb
best_params['LightGBM'] = lgb_search.best_params_
y_pred_lgb = _inv_pred(lgb.predict(X_test_df))
results['LightGBM'] = {
    'r2': r2_score(y_test, y_pred_lgb),
    'mae': mean_absolute_error(y_test, y_pred_lgb),
    'rmse': np.sqrt(mean_squared_error(y_test, y_pred_lgb))
}
print(f"   Best params: {lgb_search.best_params_}")
print(f"   R²: {results['LightGBM']['r2']:.4f}, MAE: {results['LightGBM']['mae']:.4f}, RMSE: {results['LightGBM']['rmse']:.4f}")

# 3. CatBoost with Hyperparameter Tuning
print("\n3. Training CatBoost with Hyperparameter Tuning...")
print("\n3. Training CatBoost with Hyperparameter Tuning...")
cat_base = CatBoostRegressor(random_state=42, verbose=False)
cat_params = {
    'iterations': [200, 300, 500],
    'learning_rate': [0.05, 0.1, 0.15],
    'depth': [5, 7, 9],
    'subsample': [0.8, 0.9],
    'colsample_bylevel': [0.8, 0.9],
    'reg_lambda': [1.0, 2.0, 3.0]
}
cat_search = RandomizedSearchCV(
    cat_base, cat_params, n_iter=30, cv=5,
    scoring='r2', n_jobs=-1, random_state=42, verbose=1
)
cat_search.fit(X_train_df, y_train_fit)
cat = cat_search.best_estimator_
models['CatBoost'] = cat
best_params['CatBoost'] = cat_search.best_params_
models['CatBoost'] = cat
y_pred_cat = _inv_pred(cat.predict(X_test_df))
results['CatBoost'] = {
    'r2': r2_score(y_test, y_pred_cat),
    'mae': mean_absolute_error(y_test, y_pred_cat),
    'rmse': np.sqrt(mean_squared_error(y_test, y_pred_cat))
}
print(f"   Best params: {cat_search.best_params_}")
print(f"   R²: {results['CatBoost']['r2']:.4f}, MAE: {results['CatBoost']['mae']:.4f}, RMSE: {results['CatBoost']['rmse']:.4f}")

# 4. Random Forest with Hyperparameter Tuning
print("\n4. Training Random Forest with Hyperparameter Tuning...")
print("\n4. Training Random Forest with Hyperparameter Tuning...")
rf_base = RandomForestRegressor(random_state=42, n_jobs=-1)
rf_params = {
    'n_estimators': [200, 300, 500],
    'max_depth': [8, 10, 12, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2']
}
rf_search = RandomizedSearchCV(
    rf_base, rf_params, n_iter=40, cv=5,
    scoring='r2', n_jobs=-1, random_state=42, verbose=1
)
rf_search.fit(X_train_df, y_train_fit)
rf = rf_search.best_estimator_
models['RandomForest'] = rf
best_params['RandomForest'] = rf_search.best_params_
models['RandomForest'] = rf
y_pred_rf = _inv_pred(rf.predict(X_test_df))
results['RandomForest'] = {
    'r2': r2_score(y_test, y_pred_rf),
    'mae': mean_absolute_error(y_test, y_pred_rf),
    'rmse': np.sqrt(mean_squared_error(y_test, y_pred_rf))
}
print(f"   Best params: {rf_search.best_params_}")
print(f"   R²: {results['RandomForest']['r2']:.4f}, MAE: {results['RandomForest']['mae']:.4f}, RMSE: {results['RandomForest']['rmse']:.4f}")

print("\n" + "=" * 60)
print("Model Comparison (After Hyperparameter Tuning):")
print("=" * 60)
for name, metrics in results.items():

    print(f"{name:15s} - R²: {metrics['r2']:.4f}, MAE: {metrics['mae']:.4f}, RMSE: {metrics['rmse']:.4f}")

print("=" * 60)
print("Best Hyperparameters:")
print("=" * 60)
for name, params in best_params.items():
    print(f"\n{name}:")
    for key, value in params.items():
        print(f"  {key}: {value}")
    print(f"{name:15s} - R²: {metrics['r2']:.4f}, MAE: {metrics['mae']:.4f}, RMSE: {metrics['rmse']:.4f}")

STEP 3: Model Training

1. Training XGBoost with Hyperparameter Tuning...
Fitting 5 folds for each of 40 candidates, totalling 200 fits


   Best params: {'subsample': 0.9, 'reg_lambda': 1.0, 'reg_alpha': 0.1, 'n_estimators': 300, 'max_depth': 5, 'learning_rate': 0.05, 'colsample_bytree': 0.8}
   R²: 0.9767, MAE: 2.0178, RMSE: 2.5445

2. Training LightGBM with Hyperparameter Tuning...

2. Training LightGBM with Hyperparameter Tuning...
Fitting 5 folds for each of 40 candidates, totalling 200 fits
   Best params: {'subsample': 0.8, 'reg_lambda': 1.0, 'reg_alpha': 0.5, 'num_leaves': 100, 'n_estimators': 300, 'max_depth': 5, 'learning_rate': 0.05, 'colsample_bytree': 0.8}
   R²: 0.9767, MAE: 2.0211, RMSE: 2.5458

3. Training CatBoost with Hyperparameter Tuning...

3. Training CatBoost with Hyperparameter Tuning...
Fitting 5 folds for each of 30 candidates, totalling 150 fits


## 4b. Stack tuned models

Stack the hyperparameter-tuned base models (XGBoost, LightGBM, CatBoost) with a Ridge meta-learner.

In [ ]:
# Create Stacking Ensemble
print("=" * 60)
print("STEP 4: Creating Stacking Ensemble")
print("=" * 60)

# Use top 3 models for stacking
base_models = [
    ('xgb', models['XGBoost']),
    ('lgb', models['LightGBM']),
    ('cat', models['CatBoost'])
]

# Meta-learner
meta_learner = RidgeCV(alphas=[0.1, 1.0, 10.0])

# Create stacking regressor
stacker = StackingRegressor(
    estimators=base_models,
    final_estimator=meta_learner,
    cv=5,
    n_jobs=-1
)

# Train stacking model
print("\nTraining stacking ensemble...")
stacker.fit(X_train_df, y_train_fit)

# Evaluate
y_pred_stack = _inv_pred(stacker.predict(X_test_df))
stack_results = {
    'r2': r2_score(y_test, y_pred_stack),
    'mae': mean_absolute_error(y_test, y_pred_stack),
    'rmse': np.sqrt(mean_squared_error(y_test, y_pred_stack))
}

print(f"\n✓ Stacking Ensemble Results:")
print(f"   R²: {stack_results['r2']:.4f}")
print(f"   MAE: {stack_results['mae']:.4f}")
print(f"   RMSE: {stack_results['rmse']:.4f}")

print("\n" + "=" * 60)
print("FINAL MODEL COMPARISON:")
print("=" * 60)
for name, metrics in results.items():
    print(f"{name:15s} - R²: {metrics['r2']:.4f}, MAE: {metrics['mae']:.4f}")
print(f"{'Stacking':15s} - R²: {stack_results['r2']:.4f}, MAE: {stack_results['mae']:.4f}")

STEP 4: Creating Stacking Ensemble

Training stacking ensemble...

✓ Stacking Ensemble Results:
   R²: 0.9773
   MAE: 1.9891
   RMSE: 2.5142

FINAL MODEL COMPARISON:
XGBoost         - R²: 0.9767, MAE: 2.0178
LightGBM        - R²: 0.9767, MAE: 2.0211
CatBoost        - R²: 0.9772, MAE: 1.9892
RandomForest    - R²: 0.9731, MAE: 2.1729
Stacking        - R²: 0.9773, MAE: 1.9891


## 5. Save Models

In [74]:
# Save Models
print("=" * 60)
print("STEP 5: Saving Models")
print("=" * 60)

save_dir = Path(r"D:\Projects_Main\Renewal-Upsell-Advisor\Research\Relationship\models")
save_dir.mkdir(parents=True, exist_ok=True)

# Save all tuned base models
name_to_file = {'XGBoost': 'relationship_xgb', 'LightGBM': 'relationship_lgb', 'CatBoost': 'relationship_cat', 'RandomForest': 'relationship_rf'}
for name, model in models.items():
    fname = name_to_file.get(name, name.lower().replace(' ', '_'))
    joblib.dump(model, save_dir / f"{fname}.joblib")
    print(f"✓ Saved: {name} -> {fname}.joblib")

# Save best model (stacking ensemble)
joblib.dump(stacker, save_dir / "improved_relationship_stacker.joblib")
print("✓ Saved: Best model (Stacking Ensemble) -> improved_relationship_stacker.joblib")

# Save best hyperparameters
joblib.dump(best_params, save_dir / "improved_best_params.joblib")
print("✓ Saved: Best hyperparameters -> improved_best_params.joblib")

# Save scaler
joblib.dump(scaler, save_dir / "improved_scaler.joblib")
print("✓ Saved: RobustScaler")

# Save feature names (after encoding)
feature_names = list(X_features_encoded.columns)
with open(save_dir / "improved_feature_names.json", 'w') as f:
    json.dump(feature_names, f, indent=2)
print(f"✓ Saved: Feature names ({len(feature_names)} features)")
print(f"  Features: {feature_names}")

print("\n" + "=" * 60)
print("All models saved successfully!")
print(f"Location: {save_dir}")
print("=" * 60)

STEP 5: Saving Models
✓ Saved: XGBoost -> relationship_xgb.joblib
✓ Saved: LightGBM -> relationship_lgb.joblib
✓ Saved: Best model (Stacking Ensemble) -> improved_relationship_stacker.joblib
✓ Saved: Best hyperparameters -> improved_best_params.joblib
✓ Saved: RobustScaler
✓ Saved: Feature names (13 features)
  Features: ['days_since_last_contact', 'sentiment_score', 'licenses_total', 'licenses_used', 'utilization_percentage', 'sentiment_x_utilization', 'licenses_ratio', 'days_recent', 'days_mid', 'sentiment_category_neutral', 'sentiment_category_positive', 'sentiment_category_very_negative', 'sentiment_category_very_positive']

All models saved successfully!
Location: D:\Projects_Main\Renewal-Upsell-Advisor\Research\Relationship\models
